# 03 — Entraînement LoRA

**Début** : charge `02_preprocessing`. **Fin** : checkpoint + state.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "shared").exists():
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT))

import time

import torch
from torch.optim import AdamW
from tqdm import tqdm

from shared.paths import STEP_03_CHECKPOINT
from shared.pokemon_dataset import build_loaders_from_manifest, pick_device
from shared.sd_lora_models import DEFAULT_MODEL_ID, build_sd_lora_stack
from shared.step_artifacts import load_previous_step_output, rel_path, resolve_path, save_step_output

prev = load_previous_step_output("03_train_model")
manifest_path = resolve_path(prev["manifest_path"])

train_dataset, val_dataset, train_loader, val_loader, tokenizer = build_loaders_from_manifest(manifest_path)

device = pick_device()
model_id = DEFAULT_MODEL_ID
vae, unet, text_encoder, noise_scheduler = build_sd_lora_stack(device, model_id=model_id)
CHECKPOINT_PATH = STEP_03_CHECKPOINT
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)


[02_preprocessing] État chargé ← /home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/02_preprocessing/state.json


In [2]:
import numpy as np
from torch.utils.data import Subset
from torch.utils.data import DataLoader

EPOCHS = 1
LR = 1e-5
optimizer = AdamW(unet.parameters(), lr=LR)
train_losses, val_losses, start_epoch = [], [], 0

# -----------------------------
# Use only 1/1120 of the dataset
# -----------------------------
fraction = 1 / 1120
seed = 42
batch_size = 16
num_workers = 4

def make_subset(dataset, fraction, seed):
    n = len(dataset)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n)[:int(n * fraction)]
    return Subset(dataset, indices)

train_dataset_small = make_subset(train_dataset, fraction, seed)
val_dataset_small = make_subset(val_dataset, fraction, seed + 1)

# Rebuild loaders (IMPORTANT)
train_loader = DataLoader(
    train_dataset_small,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset_small,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

# -----------------------------
# Checkpoint loading
# -----------------------------
if CHECKPOINT_PATH.exists():
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    unet.load_state_dict(ckpt["unet_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"]
    train_losses = ckpt["train_losses"]
    val_losses = ckpt["val_losses"]
    print(f"Reprise epoch {start_epoch}")
else:
    print("No checkpoint found, starting training from scratch")


def encode_prompt(te, input_ids, attention_mask):
    with torch.no_grad():
        return te(input_ids, attention_mask=attention_mask)[0]


print(f"Starting training for {EPOCHS} epochs...")

if start_epoch < EPOCHS:
    for epoch in range(start_epoch, EPOCHS):
        unet.train()
        running = 0.0

        for batch in tqdm(train_loader, desc=f"Train {epoch+1}/{EPOCHS}"):
            pv = batch["pixel_values"].to(device)
            ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)

            with torch.no_grad():
                latents = vae.encode(pv).latent_dist.sample() * 0.18215

            enc = encode_prompt(text_encoder, ids, mask)

            noise = torch.randn_like(latents)
            bsz = latents.shape[0]
            ts = torch.randint(
                0,
                noise_scheduler.config.num_train_timesteps,
                (bsz,),
                device=device
            ).long()

            noisy_latents = noise_scheduler.add_noise(latents, noise, ts)
            pred = unet(noisy_latents, ts, enc).sample

            loss = torch.nn.functional.mse_loss(pred.float(), noise.float())

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
            optimizer.step()

            running += loss.item() * bsz

        train_losses.append(running / len(train_dataset_small))

        # -----------------------------
        # Validation
        # -----------------------------
        unet.eval()
        running = 0.0

        with torch.no_grad():
            for batch in val_loader:
                pv = batch["pixel_values"].to(device)
                ids = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)

                latents = vae.encode(pv).latent_dist.sample() * 0.18215
                enc = encode_prompt(text_encoder, ids, mask)

                noise = torch.randn_like(latents)
                bsz = latents.shape[0]

                ts = torch.randint(
                    0,
                    noise_scheduler.config.num_train_timesteps,
                    (bsz,),
                    device=device
                ).long()

                noisy_latents = noise_scheduler.add_noise(latents, noise, ts)
                pred = unet(noisy_latents, ts, enc).sample

                running += torch.nn.functional.mse_loss(
                    pred.float(),
                    noise.float()
                ).item() * bsz

        val_losses.append(running / len(val_dataset_small))

        print(
            f"Epoch {epoch+1}: "
            f"train={train_losses[-1]:.4f} "
            f"val={val_losses[-1]:.4f}"
        )

        torch.save({
            "epoch": epoch + 1,
            "unet_state_dict": unet.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_losses": train_losses,
            "val_losses": val_losses,
            "model_id": model_id,
        }, CHECKPOINT_PATH)


save_step_output("03_train_model", {
    "manifest_path": prev["manifest_path"],
    "checkpoint_path": rel_path(CHECKPOINT_PATH),
    "model_id": model_id,
    "epochs_done": len(train_losses),
    "final_train_loss": train_losses[-1] if train_losses else None,
    "final_val_loss": val_losses[-1] if val_losses else None,
})

No checkpoint found, starting training from scratch
Starting training for 1 epochs...


Train 1/1: 100%|██████████| 1/1 [04:29<00:00, 269.05s/it]


Epoch 1: train=0.0740 val=0.0048
[03_train_model] État sauvegardé → /home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/03_train_model/state.json


PosixPath('/home/light/Documents/DO5/MLOps/mlops-pokegen/artifacts/pipeline/steps/03_train_model/state.json')